# Chat Models - Summarizing Chains

This notebook demonstrates three different summarization strategies using LangChain's legacy chain API: **Stuff**, **Map-Reduce**, and **Refine**.

# Chat Models - Summarizing Chains

This notebook demonstrates three different summarization strategies using LangChain's legacy chain API: **Stuff**, **Map-Reduce**, and **Refine**.

In [ ]:
%pip install langchain-classic langchain-openai langchain-community tiktoken chromadb --upgrade

In [ ]:
import getpass
import os
import warnings

# Suppress LangChain deprecation warnings (we're intentionally using legacy chains)
warnings.filterwarnings("ignore", category=DeprecationWarning, module="langchain")

# Set USER_AGENT for web requests
os.environ["USER_AGENT"] = "langchain-summarization-notebook"

# Set up your OpenAI client
if not os.getenv("OPENAI_API_KEY"):
    secret_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = secret_key

## 1. Stuff Chain - Simple Summarization

The **stuff** chain is the simplest approach: it "stuffs" all documents into a single prompt and asks the LLM to summarize them in one go. This works well when the total content fits within the model's context window.

In [ ]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_classic.chains import LLMChain
from langchain_community.document_loaders import WebBaseLoader
from langchain_classic.chains.summarize import load_summarize_chain
from langchain_core.prompts import PromptTemplate

loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

llm = ChatOpenAI(temperature=0)
chain = load_summarize_chain(llm, chain_type="stuff")

chain.invoke(docs)

ModuleNotFoundError: No module named 'langchain_openai'

## 2. Map-Reduce Chain

The **map-reduce** strategy splits documents into chunks, summarizes each chunk independently (the "map" step), then combines those summaries into a final summary (the "reduce" step). This is useful for documents that exceed the model's context window.

In [ ]:
from langchain_classic.chains import ReduceDocumentsChain, MapReduceDocumentsChain, StuffDocumentsChain
from langchain_text_splitters import CharacterTextSplitter

llm = ChatOpenAI(temperature=0)

# Map
map_template = """The following is a set of documents
{docs}
Based on this list of docs, please identify the main themes
Helpful Answer:"""
map_prompt = PromptTemplate.from_template(map_template)

# map_chain:
map_chain = LLMChain(llm=llm, prompt=map_prompt)

# Reduce
reduce_template = """The following is set of summaries:
{doc_summaries}
Take these and distill it into a final, consolidated summary of the main themes.
Helpful Answer:"""
reduce_prompt = PromptTemplate.from_template(reduce_template)

In [ ]:
# Run chain
reduce_chain = LLMChain(llm=llm, prompt=reduce_prompt)

# Takes a list of documents, combines them into a single string, and passes this to an LLMChain
combine_documents_chain = StuffDocumentsChain(
    llm_chain=reduce_chain, document_variable_name="doc_summaries"
)

# Combines and iteratively reduces the mapped documents
reduce_documents_chain = ReduceDocumentsChain(
    # This is final chain that is called.
    combine_documents_chain=combine_documents_chain,
    # If documents exceed context for `StuffDocumentsChain`
    collapse_documents_chain=combine_documents_chain,
    # The maximum number of tokens to group documents into.
    token_max=4000,
)

In [ ]:
# Combining documents by mapping a chain over them, then combining results
map_reduce_chain = MapReduceDocumentsChain(
    # Map chain
    llm_chain=map_chain,
    # Reduce chain
    reduce_documents_chain=reduce_documents_chain,
    # The variable name in the llm_chain to put the documents in
    document_variable_name="docs",
    # Return the results of the map steps in the output
    return_intermediate_steps=False,
)

text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000, chunk_overlap=0
)
split_docs = text_splitter.split_documents(docs)

In [ ]:
print(map_reduce_chain.invoke(split_docs))

## 3. Refine Chain

The **refine** strategy processes documents sequentially. It summarizes the first chunk, then refines that summary by incorporating each subsequent chunk one at a time. This produces higher-quality summaries than map-reduce but takes longer since it can't be parallelized.

In [ ]:
prompt_template = """Write a concise summary of the following:
{text}
CONCISE SUMMARY:"""
prompt = PromptTemplate.from_template(prompt_template)

refine_template = (
    "Your job is to produce a final summary\n"
    "We have provided an existing summary up to a certain point: {existing_answer}\n"
    "We have the opportunity to refine the existing summary"
    "(only if needed) with some more context below.\n"
    "------------\n"
    "{text}\n"
    "------------\n"
    "Given the new context, refine the original summary"
    "If the context isn't useful, return the original summary."
)
refine_prompt = PromptTemplate.from_template(refine_template)
chain = load_summarize_chain(
    llm=llm,
    chain_type="refine",
    question_prompt=prompt,
    refine_prompt=refine_prompt,
    return_intermediate_steps=True,
    input_key="input_documents",
    output_key="output_text",
)
result = chain.invoke({"input_documents": split_docs}, return_only_outputs=True)

# Page 1 --> Page 2 (Refine) --> Page 3 (Refine)
print(result["output_text"])

## Summary

- **Stuff Chain**: Simplest approach. This concatenates all documents into one prompt. Best for short content that fits in the context window.
- **Map-Reduce Chain**: Splits documents into chunks, summarizes each independently, then combines. Best for large documents that exceed the context window.
- **Refine Chain**: Processes chunks sequentially, refining the summary with each new chunk. Produces higher-quality results but is slower than map-reduce.

In [ ]:
%pip install langchain-classic langchain-openai langchain-community tiktoken chromadb --upgrade

In [ ]:
import getpass
import os
import warnings

# Suppress LangChain deprecation warnings (we're intentionally using legacy chains)
warnings.filterwarnings("ignore", category=DeprecationWarning, module="langchain")

# Set USER_AGENT for web requests
os.environ["USER_AGENT"] = "langchain-summarization-notebook"

# Set up your OpenAI client
if not os.getenv("OPENAI_API_KEY"):
    secret_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = secret_key

## 1. Stuff Chain - Simple Summarization

The **stuff** chain is the simplest approach: it "stuffs" all documents into a single prompt and asks the LLM to summarize them in one go. This works well when the total content fits within the model's context window.

In [ ]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_classic.chains import LLMChain
from langchain_community.document_loaders import WebBaseLoader
from langchain_classic.chains.summarize import load_summarize_chain
from langchain_core.prompts import PromptTemplate

loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

llm = ChatOpenAI(temperature=0)
chain = load_summarize_chain(llm, chain_type="stuff")

chain.invoke(docs)

## 2. Map-Reduce Chain

The **map-reduce** strategy splits documents into chunks, summarizes each chunk independently (the "map" step), then combines those summaries into a final summary (the "reduce" step). This is useful for documents that exceed the model's context window.

In [ ]:
from langchain_classic.chains import ReduceDocumentsChain, MapReduceDocumentsChain, StuffDocumentsChain
from langchain_text_splitters import CharacterTextSplitter

llm = ChatOpenAI(temperature=0)

# Map
map_template = """The following is a set of documents
{docs}
Based on this list of docs, please identify the main themes
Helpful Answer:"""
map_prompt = PromptTemplate.from_template(map_template)

# map_chain:
map_chain = LLMChain(llm=llm, prompt=map_prompt)

# Reduce
reduce_template = """The following is set of summaries:
{doc_summaries}
Take these and distill it into a final, consolidated summary of the main themes.
Helpful Answer:"""
reduce_prompt = PromptTemplate.from_template(reduce_template)

In [ ]:
# Run chain
reduce_chain = LLMChain(llm=llm, prompt=reduce_prompt)

# Takes a list of documents, combines them into a single string, and passes this to an LLMChain
combine_documents_chain = StuffDocumentsChain(
    llm_chain=reduce_chain, document_variable_name="doc_summaries"
)

# Combines and iteratively reduces the mapped documents
reduce_documents_chain = ReduceDocumentsChain(
    # This is final chain that is called.
    combine_documents_chain=combine_documents_chain,
    # If documents exceed context for `StuffDocumentsChain`
    collapse_documents_chain=combine_documents_chain,
    # The maximum number of tokens to group documents into.
    token_max=4000,
)

In [ ]:
# Combining documents by mapping a chain over them, then combining results
map_reduce_chain = MapReduceDocumentsChain(
    # Map chain
    llm_chain=map_chain,
    # Reduce chain
    reduce_documents_chain=reduce_documents_chain,
    # The variable name in the llm_chain to put the documents in
    document_variable_name="docs",
    # Return the results of the map steps in the output
    return_intermediate_steps=False,
)

text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000, chunk_overlap=0
)
split_docs = text_splitter.split_documents(docs)

In [ ]:
print(map_reduce_chain.invoke(split_docs))

## 3. Refine Chain

The **refine** strategy processes documents sequentially. It summarizes the first chunk, then refines that summary by incorporating each subsequent chunk one at a time. This produces higher-quality summaries than map-reduce but takes longer since it can't be parallelized.

In [ ]:
prompt_template = """Write a concise summary of the following:
{text}
CONCISE SUMMARY:"""
prompt = PromptTemplate.from_template(prompt_template)

refine_template = (
    "Your job is to produce a final summary\n"
    "We have provided an existing summary up to a certain point: {existing_answer}\n"
    "We have the opportunity to refine the existing summary"
    "(only if needed) with some more context below.\n"
    "------------\n"
    "{text}\n"
    "------------\n"
    "Given the new context, refine the original summary"
    "If the context isn't useful, return the original summary."
)
refine_prompt = PromptTemplate.from_template(refine_template)
chain = load_summarize_chain(
    llm=llm,
    chain_type="refine",
    question_prompt=prompt,
    refine_prompt=refine_prompt,
    return_intermediate_steps=True,
    input_key="input_documents",
    output_key="output_text",
)
result = chain.invoke({"input_documents": split_docs}, return_only_outputs=True)

# Page 1 --> Page 2 (Refine) --> Page 3 (Refine)
print(result["output_text"])

## Summary

- **Stuff Chain**: Simplest approach. This concatenates all documents into one prompt. Best for short content that fits in the context window.
- **Map-Reduce Chain**: Splits documents into chunks, summarizes each independently, then combines. Best for large documents that exceed the context window.
- **Refine Chain**: Processes chunks sequentially, refining the summary with each new chunk. Produces higher-quality results but is slower than map-reduce.

# Chat Models - Summarizing Chains

This notebook demonstrates three different summarization strategies using LangChain's legacy chain API: **Stuff**, **Map-Reduce**, and **Refine**.

In [ ]:
%pip install langchain-classic langchain-openai langchain-community tiktoken chromadb --upgrade

In [ ]:
import getpass
import os
import warnings

# Suppress LangChain deprecation warnings (we're intentionally using legacy chains)
warnings.filterwarnings("ignore", category=DeprecationWarning, module="langchain")

# Set USER_AGENT for web requests
os.environ["USER_AGENT"] = "langchain-summarization-notebook"

# Set up your OpenAI client
if not os.getenv("OPENAI_API_KEY"):
    secret_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = secret_key

## 1. Stuff Chain - Simple Summarization

The **stuff** chain is the simplest approach: it "stuffs" all documents into a single prompt and asks the LLM to summarize them in one go. This works well when the total content fits within the model's context window.

In [ ]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_classic.chains import LLMChain
from langchain_community.document_loaders import WebBaseLoader
from langchain_classic.chains.summarize import load_summarize_chain
from langchain_core.prompts import PromptTemplate

loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

llm = ChatOpenAI(temperature=0)
chain = load_summarize_chain(llm, chain_type="stuff")

chain.invoke(docs)

## 2. Map-Reduce Chain

The **map-reduce** strategy splits documents into chunks, summarizes each chunk independently (the "map" step), then combines those summaries into a final summary (the "reduce" step). This is useful for documents that exceed the model's context window.

In [ ]:
from langchain_classic.chains import ReduceDocumentsChain, MapReduceDocumentsChain, StuffDocumentsChain
from langchain_text_splitters import CharacterTextSplitter

llm = ChatOpenAI(temperature=0)

# Map
map_template = """The following is a set of documents
{docs}
Based on this list of docs, please identify the main themes
Helpful Answer:"""
map_prompt = PromptTemplate.from_template(map_template)

# map_chain:
map_chain = LLMChain(llm=llm, prompt=map_prompt)

# Reduce
reduce_template = """The following is set of summaries:
{doc_summaries}
Take these and distill it into a final, consolidated summary of the main themes.
Helpful Answer:"""
reduce_prompt = PromptTemplate.from_template(reduce_template)

In [ ]:
# Run chain
reduce_chain = LLMChain(llm=llm, prompt=reduce_prompt)

# Takes a list of documents, combines them into a single string, and passes this to an LLMChain
combine_documents_chain = StuffDocumentsChain(
    llm_chain=reduce_chain, document_variable_name="doc_summaries"
)

# Combines and iteratively reduces the mapped documents
reduce_documents_chain = ReduceDocumentsChain(
    # This is final chain that is called.
    combine_documents_chain=combine_documents_chain,
    # If documents exceed context for `StuffDocumentsChain`
    collapse_documents_chain=combine_documents_chain,
    # The maximum number of tokens to group documents into.
    token_max=4000,
)

In [ ]:
# Combining documents by mapping a chain over them, then combining results
map_reduce_chain = MapReduceDocumentsChain(
    # Map chain
    llm_chain=map_chain,
    # Reduce chain
    reduce_documents_chain=reduce_documents_chain,
    # The variable name in the llm_chain to put the documents in
    document_variable_name="docs",
    # Return the results of the map steps in the output
    return_intermediate_steps=False,
)

text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000, chunk_overlap=0
)
split_docs = text_splitter.split_documents(docs)

In [ ]:
print(map_reduce_chain.invoke(split_docs))

## 3. Refine Chain

The **refine** strategy processes documents sequentially. It summarizes the first chunk, then refines that summary by incorporating each subsequent chunk one at a time. This produces higher-quality summaries than map-reduce but takes longer since it can't be parallelized.

In [ ]:
prompt_template = """Write a concise summary of the following:
{text}
CONCISE SUMMARY:"""
prompt = PromptTemplate.from_template(prompt_template)

refine_template = (
    "Your job is to produce a final summary\n"
    "We have provided an existing summary up to a certain point: {existing_answer}\n"
    "We have the opportunity to refine the existing summary"
    "(only if needed) with some more context below.\n"
    "------------\n"
    "{text}\n"
    "------------\n"
    "Given the new context, refine the original summary"
    "If the context isn't useful, return the original summary."
)
refine_prompt = PromptTemplate.from_template(refine_template)
chain = load_summarize_chain(
    llm=llm,
    chain_type="refine",
    question_prompt=prompt,
    refine_prompt=refine_prompt,
    return_intermediate_steps=True,
    input_key="input_documents",
    output_key="output_text",
)
result = chain.invoke({"input_documents": split_docs}, return_only_outputs=True)

# Page 1 --> Page 2 (Refine) --> Page 3 (Refine)
print(result["output_text"])

## Summary

- **Stuff Chain**: Simplest approach. This concatenates all documents into one prompt. Best for short content that fits in the context window.
- **Map-Reduce Chain**: Splits documents into chunks, summarizes each independently, then combines. Best for large documents that exceed the context window.
- **Refine Chain**: Processes chunks sequentially, refining the summary with each new chunk. Produces higher-quality results but is slower than map-reduce.